In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

files = sorted(RAW_DIR.rglob("*.csv"))

files

[WindowsPath('../data/raw/서울특별시_전월세가_2023/서울특별시_전월세가_2023.csv'),
 WindowsPath('../data/raw/서울특별시_전월세가_2024/서울특별시_전월세가_2024.csv'),
 WindowsPath('../data/raw/서울특별시_전월세가_2025/서울특별시_전월세가_2025.csv')]

In [2]:
use_cols = [
    "접수년도",
    "자치구명",
    "법정동명",
    "층",
    "계약일",
    "전월세구분",
    "임대면적",
    "보증금(만원)",
    "임대료(만원)",
    "건물명",
    "건축년도",
    "건물용도",
    "계약기간",
    "신규계약구분",
]

In [4]:
apt_dfs = []

for file in files:
    loaded = False

    for encoding in ["utf-8", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(
                file,
                usecols=use_cols,
                encoding=encoding
            )

            loaded = True
            print(f"{file.name} -> encoding: {encoding}")
            break

        except UnicodeDecodeError:
            continue

    if not loaded:
        raise ValueError(f"파일을 읽을 수 없습니다: {file.name}")

    # 아파트만 필터링
    df = df[
        df["건물용도"] == "아파트"
    ].copy()

    # 원본 파일 기록
    df["source_file"] = file.name

    apt_dfs.append(df)

    print("shape:", df.shape)
    print()

서울특별시_전월세가_2023.csv -> encoding: utf-8
shape: (233849, 15)

서울특별시_전월세가_2024.csv -> encoding: cp949
shape: (202973, 15)

서울특별시_전월세가_2025.csv -> encoding: cp949
shape: (444983, 15)



In [5]:
df_apt = pd.concat(
    apt_dfs,
    ignore_index=True
)

df_apt.shape

(881805, 15)

In [6]:
df_apt["접수년도"].value_counts().sort_index()

접수년도
2023    233849
2024    202973
2025    444983
Name: count, dtype: int64

In [7]:
pd.crosstab(
    df_apt["접수년도"],
    df_apt["전월세구분"]
)

전월세구분,월세,전세
접수년도,,
2023,94542,139307
2024,83602,119366
2025,195451,249531


In [8]:
df_apt.isnull().sum().sort_values(ascending=False)

신규계약구분         84643
계약기간           79626
건축년도             177
전월세구분              6
건물명                5
법정동명               5
자치구명               5
임대면적               0
계약일                0
층                  0
접수년도               0
보증금(만원)            0
임대료(만원)            0
건물용도               0
source_file        0
dtype: int64

In [9]:
print("면적 0 이하:", (df_apt["임대면적"] <= 0).sum())
print("보증금 0 미만:", (df_apt["보증금(만원)"] < 0).sum())
print("임대료 0 미만:", (df_apt["임대료(만원)"] < 0).sum())
print("층 0 이하:", (df_apt["층"] <= 0).sum())
print("건축년도 1900 미만:", (df_apt["건축년도"] < 1900).sum())

print(
    "전세인데 월세가 0이 아닌 건수:",
    (
        (df_apt["전월세구분"] == "전세")
        & (df_apt["임대료(만원)"] != 0)
    ).sum()
)

print(
    "월세인데 월세가 0인 건수:",
    (
        (df_apt["전월세구분"] == "월세")
        & (df_apt["임대료(만원)"] == 0)
    ).sum()
)

면적 0 이하: 0
보증금 0 미만: 0
임대료 0 미만: 0
층 0 이하: 92
건축년도 1900 미만: 0
전세인데 월세가 0이 아닌 건수: 0
월세인데 월세가 0인 건수: 74


In [10]:
missing_key_rows = df_apt[
    df_apt[
        [
            "전월세구분",
            "자치구명",
            "법정동명",
            "건물명"
        ]
    ].isnull().any(axis=1)
]

missing_key_rows

,접수년도,자치구명,법정동명,층,계약일,전월세구분,임대면적,보증금(만원),임대료(만원),건물명,건축년도,건물용도,계약기간,신규계약구분,source_file
236323,2024,서대문구,남가좌동,7.0,20240307,NaN,59.442,20000,150,DMC에코자이,2019.0,아파트,24.03~26.03,신규,서울특별시_전월세가_2024.csv
244651,2024,중랑구,신내동,1.0,20240318,NaN,84.450,20000,80,데시앙,2010.0,아파트,24.05~26.05,신규,서울특별시_전월세가_2024.csv
266051,2024,마포구,공덕동,5.0,20240305,NaN,84.940,5000,250,공덕1삼성래미안,1999.0,아파트,24.05~26.05,신규,서울특별시_전월세가_2024.csv
293698,2024,양천구,신정동,4.0,20240317,NaN,71.640,52000,0,목동신시가지12,1988.0,아파트,24.05~26.02,갱신,서울특별시_전월세가_2024.csv
307918,2024,강서구,화곡동,20.0,20240308,NaN,85.000,65000,0,우장산롯데캐슬,2003.0,아파트,24.03~26.03,갱신,서울특별시_전월세가_2024.csv
451179,2025,강동구,강일동,3.0,20251205,NaN,29.960,0,29,강일리버파크11단지,2015.0,아파트,26.01~28.03,신규,서울특별시_전월세가_2025.csv
619909,2025,NaN,NaN,10.0,20250620,전세,59.985,45000,0,NaN,NaN,아파트,25.08~27.08,신규,서울특별시_전월세가_2025.csv
619910,2025,NaN,NaN,7.0,20250620,월세,59.985,30000,60,NaN,NaN,아파트,25.08~27.08,신규,서울특별시_전월세가_2025.csv
620733,2025,NaN,NaN,7.0,20250620,월세,59.985,30000,60,NaN,NaN,아파트,25.08~27.08,신규,서울특별시_전월세가_2025.csv
620734,2025,NaN,NaN,10.0,20250620,전세,59.985,45000,0,NaN,NaN,아파트,25.08~27.08,신규,서울특별시_전월세가_2025.csv


In [11]:
missing_key_rows.shape

(11, 15)

In [12]:
missing_key_rows[
    [
        "접수년도",
        "자치구명",
        "법정동명",
        "건물명",
        "전월세구분",
        "임대면적",
        "보증금(만원)",
        "임대료(만원)",
        "건축년도",
        "source_file"
    ]
].sort_values("접수년도")

,접수년도,자치구명,법정동명,건물명,전월세구분,임대면적,보증금(만원),임대료(만원),건축년도,source_file
236323,2024,서대문구,남가좌동,DMC에코자이,NaN,59.442,20000,150,2019.0,서울특별시_전월세가_2024.csv
244651,2024,중랑구,신내동,데시앙,NaN,84.450,20000,80,2010.0,서울특별시_전월세가_2024.csv
266051,2024,마포구,공덕동,공덕1삼성래미안,NaN,84.940,5000,250,1999.0,서울특별시_전월세가_2024.csv
293698,2024,양천구,신정동,목동신시가지12,NaN,71.640,52000,0,1988.0,서울특별시_전월세가_2024.csv
307918,2024,강서구,화곡동,우장산롯데캐슬,NaN,85.000,65000,0,2003.0,서울특별시_전월세가_2024.csv
451179,2025,강동구,강일동,강일리버파크11단지,NaN,29.960,0,29,2015.0,서울특별시_전월세가_2025.csv
619909,2025,NaN,NaN,NaN,전세,59.985,45000,0,NaN,서울특별시_전월세가_2025.csv
619910,2025,NaN,NaN,NaN,월세,59.985,30000,60,NaN,서울특별시_전월세가_2025.csv
620733,2025,NaN,NaN,NaN,월세,59.985,30000,60,NaN,서울특별시_전월세가_2025.csv
620734,2025,NaN,NaN,NaN,전세,59.985,45000,0,NaN,서울특별시_전월세가_2025.csv


In [13]:
df_apt[
    [
        "임대면적",
        "층",
        "보증금(만원)",
        "임대료(만원)",
        "건축년도"
    ]
].describe()

,임대면적,층,보증금(만원),임대료(만원),건축년도
count,881805.000000,881805.000000,881805.000000,881805.000000,881628.000000
mean,68.823969,9.633032,41383.149779,46.141049,2005.036047
std,28.906017,6.478712,36369.191628,96.466241,12.623524
min,9.040000,-4.000000,0.000000,0.000000,1961.000000
25%,49.993000,5.000000,13000.000000,0.000000,1995.000000
50%,59.990000,9.000000,36000.000000,0.000000,2005.000000
75%,84.920000,13.000000,59500.000000,60.000000,2017.000000
max,317.360000,69.000000,1000000.000000,9600.000000,2025.000000


In [14]:
before_count = len(df_apt)

before_count

881805

In [15]:
essential_cols = [
    "자치구명",
    "법정동명",
    "건물명",
    "전월세구분",
]

df_clean = df_apt.dropna(
    subset=essential_cols
).copy()

df_clean.shape

(881794, 15)

In [16]:
removed_count = before_count - len(df_clean)

print("제거 전:", before_count)
print("제거 후:", len(df_clean))
print("제거 건수:", removed_count)

제거 전: 881805
제거 후: 881794
제거 건수: 11


In [17]:
df_clean["valid_monthly_rent"] = ~(
    (df_clean["전월세구분"] == "월세")
    & (df_clean["임대료(만원)"] == 0)
)

df_clean["valid_monthly_rent"].value_counts()

valid_monthly_rent
True     881720
False        74
Name: count, dtype: int64

In [18]:
df_monthly = df_clean[
    (df_clean["전월세구분"] == "월세")
    & (df_clean["valid_monthly_rent"])
]

In [19]:
df_clean["has_build_year"] = df_clean["건축년도"].notna()

df_clean["has_build_year"].value_counts()

has_build_year
True     881622
False       172
Name: count, dtype: int64

In [20]:
print("전체 데이터:", len(df_clean))
print(
    "전세:",
    (df_clean["전월세구분"] == "전세").sum()
)
print(
    "월세:",
    (df_clean["전월세구분"] == "월세").sum()
)
print(
    "월세 분석 제외:",
    (~df_clean["valid_monthly_rent"]).sum()
)
print(
    "건축년도 결측:",
    (~df_clean["has_build_year"]).sum()
)

전체 데이터: 881794
전세: 508201
월세: 373593
월세 분석 제외: 74
건축년도 결측: 172


In [21]:
df_clean.dtypes

접수년도                    int64
자치구명                      str
법정동명                      str
층                     float64
계약일                     int64
전월세구분                     str
임대면적                  float64
보증금(만원)                 int64
임대료(만원)                 int64
건물명                       str
건축년도                  float64
건물용도                      str
계약기간                      str
신규계약구분                    str
source_file               str
valid_monthly_rent       bool
has_build_year           bool
dtype: object

In [22]:
df_clean["계약일"] = (
    df_clean["계약일"]
    .astype(str)
    .str.zfill(8)
)

df_clean["contract_date"] = pd.to_datetime(
    df_clean["계약일"],
    format="%Y%m%d",
    errors="coerce"
)

df_clean["contract_date"].isna().sum()

np.int64(0)

In [23]:
df_clean["contract_year"] = df_clean["contract_date"].dt.year
df_clean["contract_month"] = df_clean["contract_date"].dt.month

In [24]:
df_clean["building_age"] = (
    df_clean["contract_year"] - df_clean["건축년도"]
)

In [25]:
df_clean["pyeong"] = (
    df_clean["임대면적"] / 3.3058
)

In [26]:
df_clean["deposit_per_m2"] = (
    df_clean["보증금(만원)"] / df_clean["임대면적"]
)

In [27]:
print(
    "날짜 변환 실패:",
    df_clean["contract_date"].isna().sum()
)

print(
    "건축연식 음수:",
    (df_clean["building_age"] < 0).sum()
)

print(
    "평수 0 이하:",
    (df_clean["pyeong"] <= 0).sum()
)

print(
    "단위면적 보증금 0 미만:",
    (df_clean["deposit_per_m2"] < 0).sum()
)

날짜 변환 실패: 0
건축연식 음수: 293
평수 0 이하: 0
단위면적 보증금 0 미만: 0


In [28]:
df_clean[
    [
        "계약일",
        "contract_date",
        "contract_year",
        "contract_month",
        "건축년도",
        "building_age",
        "임대면적",
        "pyeong",
        "보증금(만원)",
        "deposit_per_m2"
    ]
].head()

,계약일,contract_date,contract_year,contract_month,건축년도,building_age,임대면적,pyeong,보증금(만원),deposit_per_m2
0,20230515,2023-05-15,2023,5,2003.0,20.0,84.71,25.624660,46000,543.029158
1,20230515,2023-05-15,2023,5,2003.0,20.0,101.97,30.845786,46500,456.016475
2,20230515,2023-05-15,2023,5,2004.0,19.0,133.03,40.241394,70000,526.197098
3,20230515,2023-05-15,2023,5,2006.0,17.0,84.94,25.694234,37500,441.488109
4,20230515,2023-05-15,2023,5,2012.0,11.0,84.98,25.706334,70000,823.723229


In [29]:
negative_age = df_clean[
    df_clean["building_age"] < 0
].copy()

negative_age[
    [
        "접수년도",
        "계약일",
        "contract_date",
        "자치구명",
        "법정동명",
        "건물명",
        "건축년도",
        "building_age",
        "전월세구분",
        "임대면적",
        "보증금(만원)",
        "임대료(만원)"
    ]
].head(30)

,접수년도,계약일,contract_date,자치구명,법정동명,건물명,건축년도,building_age,전월세구분,임대면적,보증금(만원),임대료(만원)
32334,2023,20221231,2022-12-31,은평구,증산동,DMC센트럴자이(3단지),2023.0,-1.0,전세,84.85,70000,0
48345,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,8900,33
48346,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,9500,18
48347,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,8900,33
48348,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,9500,18
48349,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,9500,18
48350,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,11300,22
48351,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,9500,18
48352,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,8900,33
48353,2023,20221223,2022-12-23,동대문구,휘경동,회기역하트리움역세권청년주택,2023.0,-1.0,월세,15.89,11300,22


In [30]:
negative_age["building_age"].value_counts().sort_index()

building_age
-1.0    293
Name: count, dtype: int64

In [31]:
negative_age[
    ["contract_year", "건축년도"]
].value_counts().sort_index()

contract_year  건축년도  
2022           2023.0     82
2023           2024.0      3
2024           2025.0    208
Name: count, dtype: int64

In [32]:
negative_age[
    [
        "자치구명",
        "법정동명",
        "건물명",
        "contract_year",
        "건축년도",
        "building_age"
    ]
].sort_values(
    "building_age"
).head(20)

,자치구명,법정동명,건물명,contract_year,건축년도,building_age
32334,은평구,증산동,DMC센트럴자이(3단지),2022,2023.0,-1.0
48345,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48346,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48347,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48348,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48349,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48350,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48351,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48352,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0
48353,동대문구,휘경동,회기역하트리움역세권청년주택,2022,2023.0,-1.0


In [33]:
df_clean["building_age_raw"] = df_clean["building_age"]

In [34]:
df_clean["pre_completion_contract"] = (
    df_clean["building_age_raw"] < 0
)

In [35]:
df_clean["building_age"] = (
    df_clean["building_age_raw"]
    .clip(lower=0)
)

In [36]:
print(
    "입주 전 계약:",
    df_clean["pre_completion_contract"].sum()
)

print(
    "보정 후 음수 연식:",
    (df_clean["building_age"] < 0).sum()
)

입주 전 계약: 293
보정 후 음수 연식: 0


In [37]:
df_clean["신규계약구분"] = (
    df_clean["신규계약구분"]
    .fillna("미상")
)

In [38]:
df_clean["신규계약구분"].value_counts(dropna=False)

신규계약구분
신규    501141
갱신    295569
미상     84643
1        368
2         73
Name: count, dtype: int64

In [39]:
print("전체 행:", len(df_clean))

print(
    "핵심 컬럼 결측:",
    df_clean[
        [
            "자치구명",
            "법정동명",
            "건물명",
            "전월세구분",
            "임대면적",
            "보증금(만원)",
            "임대료(만원)"
        ]
    ].isna().sum().sum()
)

print(
    "날짜 변환 실패:",
    df_clean["contract_date"].isna().sum()
)

print(
    "음수 건축연식:",
    (df_clean["building_age"] < 0).sum()
)

print(
    "입주 전 계약:",
    df_clean["pre_completion_contract"].sum()
)

print(
    "월세 분석 제외:",
    (~df_clean["valid_monthly_rent"]).sum()
)

전체 행: 881794
핵심 컬럼 결측: 0
날짜 변환 실패: 0
음수 건축연식: 0
입주 전 계약: 293
월세 분석 제외: 74


In [40]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [41]:
output_path = (
    PROCESSED_DIR
    / "seoul_apartment_rental_2023_2025_clean.csv"
)

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(output_path)

..\data\processed\seoul_apartment_rental_2023_2025_clean.csv


In [42]:
output_path.exists()

True

In [43]:
print(
    "파일 크기(MB):",
    round(output_path.stat().st_size / 1024 / 1024, 2)
)

파일 크기(MB): 197.65
